In [18]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.read_csv("data/train_data.csv")
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_renovated,lat,long,hous_val_amt,trend,trend_sq,house_age,was_renovated
0,7129300520,2014-10-13,221900.0,3,1,1180,5650,1.0,0,0,...,1180,0,0,47.5112,-122.257,175400.0,5,25,59,False
1,6414100192,2014-12-09,538000.0,3,2,2570,7242,2.0,0,0,...,2170,400,1991,47.7210,-122.319,225900.0,7,49,63,True
2,5631500400,2015-02-25,180000.0,2,1,770,10000,1.0,0,0,...,770,0,0,47.7379,-122.233,246600.0,9,81,82,False
3,2487200875,2014-12-09,604000.0,4,3,1960,5000,1.0,0,0,...,1050,910,0,47.5208,-122.393,280200.0,7,49,49,False
4,1954400510,2015-02-18,510000.0,3,2,1680,8080,1.0,0,0,...,1680,0,0,47.6168,-122.045,239850.0,9,81,28,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18559,263000018,2014-05-21,360000.0,3,2,1530,1131,3.0,0,0,...,1530,0,0,47.6993,-122.346,279200.0,0,0,5,False
18560,6600060120,2015-02-23,400000.0,4,2,2310,5813,2.0,0,0,...,2310,0,0,47.5107,-122.362,180100.0,9,81,1,False
18561,1523300141,2014-06-23,402101.0,2,1,1020,1350,2.0,0,0,...,1020,0,0,47.5944,-122.299,219400.0,1,1,5,False
18562,291310100,2015-01-16,400000.0,3,2,1600,2388,2.0,0,0,...,1600,0,0,47.5345,-122.069,308700.0,8,64,11,False


In [3]:
features = ["bedrooms", "bathrooms",  "floors", "sqft_living", "sqft_lot", "sqft_above", "house_age"]
target = "price"

In [ ]:
work_df = df[features].copy()
work_df["target"] = df[target]
work_df["pred_y"] = work_df["target"].mean()
lista = len(work_df) *[2]
work_df["pred_y"]*lista


In [14]:
def XGBoost(set_df, features, target, lambida, gama, min_split, min_leaf, cicles, rate):

  work_df = set_df[features].copy()
  work_df["target"] = set_df[target]

  work_df["hessian"] = 2
  work_df["pred_y"] = work_df["target"].mean()
  work_df["gradient"] = (-2)*(work_df["target"] - work_df["pred_y"])
  SQRES = abs(work_df["pred_y"] - work_df["target"]).mean()
  model = list()
  SQRES_list = [SQRES]
  for i in range(0, cicles):

    tree = node_function(work_df, features, gama, lambida, min_split, min_leaf)
    model.append(tree)
    weights = list()
    for s in work_df.index:
      weights.append(predict_func(tree, work_df.loc[s]))

    weights = np.array(weights)
    work_df["pred_y"] = work_df["pred_y"] + rate*weights
    work_df["gradient"] = (-2)*(work_df["target"] - work_df["pred_y"])
    SQRES_list.append(abs(work_df["pred_y"] - work_df["target"]).mean())

  return model, SQRES_list, work_df["target"].mean()

In [9]:
def node_function(data, features, gama, lambida, min_split, min_leaf):

    G = data["gradient"].sum()
    H = data["hessian"].sum()

    if len(data) <= min_split:
        return {"type": "leaf", "target_mean": data["target"].mean(), "weight": (-G)/(H + lambida)}

    top_feature = None
    top_limiar = None
    gain = -float("inf")
    K = int(len(features)*0.7)
    sorted = np.random.choice(features, size = K, replace = False)
    for i in sorted:
        temp_limiars = data[i].unique()
        if len(temp_limiars) <=20:
            np.sort(temp_limiars)
            limiars =  [(temp_limiars[s] + temp_limiars[s+1])/2 for s in range(0,len(temp_limiars)-1)]
            
        else:
            limiars = np.linspace(data[i].min(), data[i].max(), 20)

        for j in limiars:
            left_mask = data[i] <= j
            right_mask = data[i] > j
            left = data[left_mask]
            right = data[right_mask]

            if len(left) < min_leaf or len(right) < min_leaf:
                continue

            Gl = left["gradient"].sum()
            Hl = left["hessian"].sum()
            Gr = right["gradient"].sum()
            Hr = right["hessian"].sum()

            temp_gain = (((Gl**2)/(Hl + lambida)) + ((Gr**2)/(Hr + lambida)) - ((G**2)/(H + lambida)))

            if temp_gain > gain:
                gain = temp_gain
                top_feature = i
                top_limiar = j

    if gain <= gama:
        return {"type": "leaf", "target_mean": data["target"].mean(), "weight": (-G)/(H + lambida)}

    left = data[data[top_feature] <= top_limiar]
    right = data[data[top_feature] > top_limiar]

    left_child = node_function(left, features, gama, lambida, min_split, min_leaf)
    right_child = node_function(right, features, gama, lambida, min_split, min_leaf)

    return {"type": "node", "left_child": left_child, "right_child": right_child, "top_feature": top_feature, "treshold": top_limiar}

In [7]:
def predict_func(tree, sample):
  if tree["type"] == "leaf":
    return tree["weight"]

  feature = tree["top_feature"]
  treshold = tree["treshold"]

  if sample[feature] <= treshold:
    return predict_func(tree["left_child"], sample)

  if sample[feature] > treshold:
    return predict_func(tree["right_child"], sample)

In [22]:
json.dump({"model": modelo, "pregressao": progerssao, "base_score": base_score}, open("modelo_XGB.json", "w"))

In [16]:
modelo, progerssao, base_score = XGBoost(df,  features,  target,  1,  0,  5,  2,  20, 0.2)

In [17]:
progerssao

[233167.0537519568,
 196945.82024656114,
 167528.226841565,
 143510.5359261237,
 123786.11379636936,
 107759.06322978155,
 94426.71675093807,
 83356.57013765402,
 73852.86767622163,
 65772.29462321964,
 58852.37984105056,
 52935.294554369495,
 47735.64245132332,
 43235.25827269682,
 39353.042189055006,
 35860.802801046775,
 32756.79418209549,
 29954.613557087556,
 27489.326082110696,
 25208.37304817754,
 23130.62792760264]

In [12]:
modelo

{'type': 'node',
 'left_child': {'type': 'node',
  'left_child': {'type': 'node',
   'left_child': {'type': 'node',
    'left_child': {'type': 'node',
     'left_child': {'type': 'node',
      'left_child': {'type': 'node',
       'left_child': {'type': 'node',
        'left_child': {'type': 'node',
         'left_child': {'type': 'leaf',
          'target_mean': 220000.0,
          'weight': -37095.5065442285},
         'right_child': {'type': 'node',
          'left_child': {'type': 'leaf',
           'target_mean': 307500.0,
           'weight': 7559.910214451467},
          'right_child': {'type': 'node',
           'left_child': {'type': 'node',
            'left_child': {'type': 'node',
             'left_child': {'type': 'node',
              'left_child': {'type': 'node',
               'left_child': {'type': 'node',
                'left_child': {'type': 'node',
                 'left_child': {'type': 'node',
                  'left_child': {'type': 'node',
                   